## Supplied-data update — 4 November 2026
This notebook retains its previously executed analysis and source-access record. The subsequently uploaded BSOS CSV and codebooks are now analysed separately in **06_BSOS_Anxiety_Measurement.ipynb** and Appendix J. Earlier statements that BSOS data or empirical GAD-7 scores were unavailable are historical, not the current project status. The new file still lacks personalisation, comparison and digital-literacy variables. It is not merged with this notebook’s respondents.

# The Impact of Perceived Social Media Algorithm Personalisation on Anxiety Levels Among Generation Z: The Mediating Role of Social Comparison

## Notebook 03 — MindSET access and descriptive NLP

**Repaired workflow; real MindSET analysis is still awaiting authorised data.**

The public Hugging Face repository listed only `.gitattributes` when rechecked on 19 October 2026. The authors explain in **section 7.1 of their paper** that access is available on request under a **Data Use Agreement (DUA)**. An empty public page is therefore not a Python download error. This notebook checks the current public listing and accepts an authorised existing release; it does not bypass the access process.

The original proposal topic and secondary-only design are unchanged. This is a supporting descriptive text analysis, not a test of perceived personalisation, mediation, clinical anxiety or South African Generation Z. No substitute corpus or simulated research results are included.

### How to use this notebook in Colab

1. Obtain the MindSET release, its codebook and applicable DUA through the authors' contact details in the [paper](https://arxiv.org/html/2511.20672v1#S7.SS1). Ask which existing release and partitions are authorised for this secondary analysis. No request has been sent on your behalf.
2. Upload the authorised file using Colab's **Files** sidebar. For a small file, the optional upload cell below opens the file picker. For a large corpus, use an authorised local or Drive path consistent with the DUA; do not make the file public.
3. Set `DATA_PATH`, inspect the column names, and map the three fields and exact anxiety-group code using the release codebook. Record the release, codebook and access basis. Do not use guessed labels.
4. Set `ACCESS_AUTHORISED` to `True` only when that release is authorised. Restart the runtime and run all cells after any configuration changes.
5. Read `run_status.json`. Only `EXECUTED_DESCRIPTIVE_NLP` means the real-data NLP ran. An access check or a successful software test is not a study finding.

Supported inputs are flat, post-level **CSV, TSV, JSONL or Parquet**. Nested user histories require a documented conversion first. The entire file is scanned twice; only user keys and selected texts are retained in memory, rather than loading every post at once. No raw text or user identifiers are printed or exported.


### 1. Prepare the sentiment dependency

Colab normally includes NumPy. VADER is pinned to the same version as the original workflow. For Parquet input, also run `%pip install pyarrow` in a separate cell if it is absent. Dependency installation does not download the MindSET corpus.

In [1]:
import importlib.util, importlib.metadata, subprocess, sys
if importlib.util.find_spec("vaderSentiment") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vaderSentiment==3.3.2", "--quiet"])
if importlib.metadata.version("vaderSentiment") != "3.3.2":
    raise RuntimeError("Install vaderSentiment==3.3.2 and restart the runtime for a consistent analysis.")
print("VADER", importlib.metadata.version("vaderSentiment"), "ready. No corpus downloaded.")

VADER 3.3.2 ready. No corpus downloaded.


### 2. Load the workflow functions

This cell defines the loaders, validation, reproducible sampling, sentiment summaries and aggregate export. It does not run the corpus analysis.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, importlib.metadata, json, math, re, sys
import urllib.request, urllib.error
import numpy as np

PAPER_URL = 'https://arxiv.org/html/2511.20672v1#S7.SS1'
DATASET_URL = 'https://huggingface.co/datasets/smankarious/mindset'
API_URL = 'https://huggingface.co/api/datasets/smankarious/mindset'
ALGORITHM_PATTERN = r'\b(?:algorithm(?:s|ic)?|recommendation(?:s)?|personalized|personalised|for you page)\b'

def check_public_source(timeout=15):
    """Check metadata only. An HTTP/network failure is not evidence of an empty corpus."""
    audit = {'checked_at_utc': datetime.now(timezone.utc).isoformat(),
             'source': DATASET_URL, 'access_route': 'Authors; Data Use Agreement (paper section 7.1)'}
    try:
        req = urllib.request.Request(API_URL, headers={'User-Agent': 'MindSET-research-access-check/1.0'})
        with urllib.request.urlopen(req, timeout=timeout) as response:
            data = json.load(response)
        names = [x['rfilename'] for x in data.get('siblings', [])]
        candidates = [n for n in names if n.lower().endswith(('.csv', '.tsv', '.parquet', '.jsonl', '.jsonl.gz', '.zip', '.tar.gz'))]
        audit.update(status='FILES_LISTED_REVIEW_REQUIRED' if candidates else 'NO_PUBLIC_DATA_FILES_LISTED',
                     repository_revision=data.get('sha'), candidate_files=candidates,
                     note='A file listing is not permission to use a release; verify its documentation and access terms.')
    except urllib.error.HTTPError as error:
        audit.update(status='ACCESS_RESTRICTED' if error.code in (401, 403) else 'SOURCE_CHECK_FAILED',
                     http_status=error.code)
    except (OSError, ValueError, KeyError):
        audit.update(status='SOURCE_CHECK_FAILED', note='Network or metadata error; availability is unknown. Retry later.')
    return audit

def scalar(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    if isinstance(value, (list, dict, tuple, set)):
        raise ValueError('Nested or multi-label fields require a documented conversion; no automatic mapping was made.')
    return str(value).strip()

def row_stream(path):
    """Read flat post-level CSV/TSV/JSONL or Parquet without loading the full corpus."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in ('.csv', '.tsv'):
        csv.field_size_limit(min(sys.maxsize, 2**31 - 1))
        with path.open(encoding='utf-8-sig', newline='') as handle:
            reader = csv.DictReader(handle, delimiter='\t' if suffix == '.tsv' else ',')
            if not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames):
                raise ValueError('Missing or duplicate column names. Supply a documented flat table.')
            for row in reader:
                if None in row:
                    raise ValueError('A delimited row has extra fields. Check quoting and delimiter; raw content is not displayed.')
                yield row
    elif suffix == '.jsonl':
        with path.open(encoding='utf-8-sig') as handle:
            for number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    raise ValueError(f'Invalid JSON on line {number}; raw content is not displayed.') from None
                if not isinstance(row, dict):
                    raise ValueError('Each JSONL line must be one flat post record.')
                yield row
    elif suffix == '.parquet':
        try:
            import pyarrow.parquet as pq
        except ImportError:
            raise ImportError('Parquet needs pyarrow. Run %pip install pyarrow, then rerun this notebook.') from None
        for batch in pq.ParquetFile(path).iter_batches(batch_size=10000):
            yield from batch.to_pylist()
    else:
        raise ValueError('Supported inputs: .csv, .tsv, .jsonl and .parquet. Extract an authorised archive first.')

def inspect_columns(path):
    first = next(row_stream(path), None)
    if first is None:
        raise ValueError('The input contains no records.')
    return list(first.keys())

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def digest_parts(*values):
    return hashlib.sha256(json.dumps(values, ensure_ascii=False).encode('utf-8')).hexdigest()

def select_users(path, text_column, user_column, label_column, anxiety_label, seed=8412, max_users=5000):
    """Sample users first, then one distinct-text post per user; never sample prolific users more often."""
    columns = [text_column, user_column, label_column]
    if any(not isinstance(c, str) or not c.strip() for c in columns) or len(set(columns)) != 3:
        raise ValueError('Map three distinct non-empty column names using the release codebook.')
    if not isinstance(anxiety_label, str) or not anxiety_label.strip():
        raise ValueError('Set the exact anxiety-group label as a string from the codebook; no label is inferred.')
    if not 1 <= max_users <= 5000:
        raise ValueError('MAX_USERS must be between 1 and the planned cap of 5000.')
    if not set(columns).issubset(inspect_columns(path)):
        raise ValueError('Mapped columns are absent. Inspect column names and correct the configuration.')
    source_hash = file_sha256(path)
    counts = dict(source_rows=0, missing_label=0, other_label=0, anxiety_rows=0,
                  unusable_text_or_user=0, eligible_rows=0)
    users = set()
    invalid = {'', '[deleted]', '[removed]'}

    def eligible(row):
        if any(c not in row for c in columns):
            raise ValueError('Some records are missing mapped fields; repair the documented export before analysis.')
        label = scalar(row[label_column])
        text = scalar(row[text_column])
        user = scalar(row[user_column])
        return label, text, user

    for row in row_stream(path):
        counts['source_rows'] += 1
        label, text, user = eligible(row)
        if not label:
            counts['missing_label'] += 1
        elif label != anxiety_label.strip():
            counts['other_label'] += 1
        else:
            counts['anxiety_rows'] += 1
            if text.casefold() in invalid or user.casefold() in invalid:
                counts['unusable_text_or_user'] += 1
            else:
                counts['eligible_rows'] += 1
                users.add(user)
    if not users:
        raise ValueError('No eligible users. Check the exact label, field mapping and source documentation; do not invent records.')
    # Seeded hash priorities are independent of posting frequency and file order.
    chosen = sorted(users, key=lambda u: (digest_parts(seed, 'user', u), u))[:max_users]
    chosen_set = set(chosen)
    best = {}
    for row in row_stream(path):
        label, text, user = eligible(row)
        if label == anxiety_label.strip() and user in chosen_set and text.casefold() not in invalid:
            key = digest_parts(seed, 'post', user, text)
            if user not in best or key < best[user][0]:
                best[user] = (key, text)
    if file_sha256(path) != source_hash:
        raise ValueError('Input changed during the two-pass read. Restart using an unchanged source file.')
    # Remove repeated selected texts across users, keeping the first seeded user.
    seen, texts = set(), []
    for user in chosen:
        if user not in best:
            raise ValueError('A selected user vanished between passes; check source consistency.')
        text = best[user][1]
        key = digest_parts(text)
        if key not in seen:
            seen.add(key)
            texts.append(text)
    counts.update(eligible_users=len(users), sampled_users_before_text_dedup=len(chosen),
                  duplicate_selected_texts_removed=len(chosen) - len(texts), analysed_users=len(texts))
    return texts, counts, source_hash

def summarise_sentiment(texts, seed=8412, resamples=2000):
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyser = SentimentIntensityAnalyzer()
    pattern = re.compile(ALGORITHM_PATTERN, re.I)
    groups = {False: [], True: []}
    for text in texts:
        groups[bool(pattern.search(text))].append(analyser.polarity_scores(text)['compound'])
    summaries = []
    for key in (False, True):
        values = np.asarray(groups[key], dtype=float)
        summaries.append({'algorithm_term': key, 'n_users': len(values),
                          'mean': float(np.mean(values)) if len(values) >= 20 else None,
                          'median': float(np.median(values)) if len(values) >= 20 else None,
                          'sample_sd': float(np.std(values, ddof=1)) if len(values) >= 20 else None})
    contrast = None
    if min(map(len, groups.values())) >= 20:
        rng = np.random.default_rng(seed)
        a, b = np.asarray(groups[True]), np.asarray(groups[False])
        differences = [rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
                       for _ in range(resamples)]
        low, high = np.quantile(differences, [.025, .975])
        contrast = {'mean_difference_term_minus_no_term': float(a.mean() - b.mean()),
                    'percentile_95_low': float(low), 'percentile_95_high': float(high),
                    'bootstrap_resamples': resamples,
                    'interpretation': 'Exploratory sentiment contrast conditional on the selected posts; not an anxiety or causal effect.'}
    return summaries, contrast

def run_analysis(config, output_dir):
    """Write truthful run status; outputs never include raw text or user identifiers."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    if any(output_dir.iterdir()):
        raise ValueError('Use a new empty output directory for each run to prevent stale results.')
    status = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'empirical_analysis_executed': False}
    def save_status():
        (output_dir / 'run_status.json').write_text(json.dumps(status, indent=2), encoding='utf-8')
    if not config.get('access_authorised'):
        status.update(status='NOT_EXECUTED_ACCESS_REQUIRED',
                      next_step='Obtain the existing MindSET release and Data Use Agreement from its authors; see paper section 7.1.')
        save_status()
        return status
    if not config.get('data_path'):
        status.update(status='NOT_EXECUTED_INPUT_REQUIRED', next_step='Upload the authorised file and set DATA_PATH.')
        save_status()
        return status
    try:
        for key in ('release', 'access_reference', 'codebook_reference'):
            if not str(config.get(key, '')).strip():
                raise ValueError('Record the release version, access basis and codebook reference before analysis.')
        path = Path(config['data_path'])
        if not path.is_file():
            raise ValueError('DATA_PATH does not identify an existing file.')
        texts, counts, source_hash = select_users(path, config['text_column'], config['user_column'],
                                               config['label_column'], config['anxiety_label'],
                                               config['seed'], config['max_users'])
        summaries, contrast = summarise_sentiment(texts, seed=config['seed'])
        del texts
        with (output_dir / 'sentiment_summary.csv').open('w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(summaries[0]))
            writer.writeheader(); writer.writerows(summaries)
        if contrast is not None:
            (output_dir / 'sentiment_contrast.json').write_text(json.dumps(contrast, indent=2), encoding='utf-8')
        status.update(status='EXECUTED_DESCRIPTIVE_NLP', empirical_analysis_executed=True,
                      source_sha256=source_hash, release=config['release'],
                      codebook_reference=config['codebook_reference'], access_reference=config['access_reference'],
                      mapped_columns={k: config[k] for k in ('text_column', 'user_column', 'label_column')},
                      anxiety_label=config['anxiety_label'], seed=config['seed'], max_users=config['max_users'],
                      flow=counts, algorithm_pattern=ALGORITHM_PATTERN,
                      contrast_estimated=contrast is not None,
                      versions={'python': sys.version.split()[0], 'numpy': np.__version__,
                                'vaderSentiment': importlib.metadata.version('vaderSentiment')},
                      interpretation='Sentiment is not anxiety. Algorithm-related words do not measure personalisation or exposure. H1–H3 remain untested.')
        save_status()
        return status
    except Exception as error:
        status.update(status='FAILED_VALIDATION_OR_ANALYSIS', error_type=type(error).__name__,
                      next_step='Correct the indicated configuration or dependency problem and run again into a new directory.')
        # Remove any partial aggregate files so this directory cannot look like a completed run.
        for name in ('sentiment_summary.csv', 'sentiment_contrast.json'):
            (output_dir / name).unlink(missing_ok=True)
        save_status()
        raise


### 3. Check the original public source

This reads repository metadata only. A timeout is reported as unknown availability, not as proof that the corpus is empty. Newly listed files still require review against the authors’ release and DUA.

In [3]:
PUBLIC_SOURCE_AUDIT = check_public_source()
print(json.dumps(PUBLIC_SOURCE_AUDIT, indent=2))

{
  "checked_at_utc": "2026-11-04T18:19:53.243949+00:00",
  "source": "https://huggingface.co/datasets/smankarious/mindset",
  "access_route": "Authors; Data Use Agreement (paper section 7.1)",
  "status": "NO_PUBLIC_DATA_FILES_LISTED",
  "repository_revision": "10bf752aa9bbb52e2982cc82f79e574ac3ccc7ac",
  "candidate_files": [],
  "note": "A file listing is not permission to use a release; verify its documentation and access terms."
}


### 4. Upload an authorised file and set its path

The optional picker is for modest-sized files and only runs when enabled. For large files, use the Files sidebar or a permitted mounted path instead. The notebook does not mount Drive automatically.

In [4]:
OPEN_COLAB_UPLOAD = False
UPLOADED_PATH = ""
if OPEN_COLAB_UPLOAD:
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError("The upload picker requires Colab. Outside Colab, set DATA_PATH to a local file.") from None
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one documented flat data file.")
    UPLOADED_PATH = str(Path(next(iter(uploaded))).resolve())
    del uploaded
    print("Upload received. Its path is available in UPLOADED_PATH.")
else:
    print("Optional upload picker is off. Set DATA_PATH to your authorised existing file.")

Optional upload picker is off. Set DATA_PATH to your authorised existing file.


In [5]:
DATA_PATH = UPLOADED_PATH or ""  # e.g. /content/your_authorised_release.csv
if DATA_PATH:
    print("Column names only:", inspect_columns(DATA_PATH))
else:
    print("No input selected. No participant records have been read.")

No input selected. No participant records have been read.


### 5. Map the release and record its provenance

Use a stable de-identified **user** key, not a post ID. The anxiety label must be an existing codebook value; it is never inferred from sentiment. Numeric labels should be entered as their exact text representation, such as `"1"` only if the codebook specifies that value. Enter references or descriptions below, not passwords, access tokens or the text of private agreements.

In [6]:
TEXT_COLUMN = ""
USER_COLUMN = ""
LABEL_COLUMN = ""
ANXIETY_LABEL = ""
DATA_RELEASE = ""
CODEBOOK_REFERENCE = ""
ACCESS_REFERENCE = ""
ACCESS_AUTHORISED = False
SEED = 8412
MAX_USERS = 5000

CONFIG = dict(data_path=DATA_PATH, text_column=TEXT_COLUMN, user_column=USER_COLUMN,
              label_column=LABEL_COLUMN, anxiety_label=ANXIETY_LABEL,
              release=DATA_RELEASE, codebook_reference=CODEBOOK_REFERENCE,
              access_reference=ACCESS_REFERENCE, access_authorised=ACCESS_AUTHORISED,
              seed=SEED, max_users=MAX_USERS)
print("Configuration loaded; analysis has not run yet.")

Configuration loaded; analysis has not run yet.


### 6. Run validation and analysis

The planned cap remains 5,000 users and the algorithm-related term list is unchanged. **Sampling is corrected:** users are selected first with seeded hash priorities, then one distinct-text post per selected user. Repeated identical posts do not increase a user's selection probability. Duplicate selected texts across users are removed and the resulting user loss is reported, without replacing those users. The result describes selected posts, not each user's complete language history.

Missing labels, unmatched labels, blank/deleted text, unusable identifiers and user counts are recorded. Groups with fewer than 20 selected users receive counts only; the sentiment contrast is estimated only when both groups reach 20. A 2,000-resample percentile bootstrap describes uncertainty conditional on the selected posts. It does not account for corpus selection, label error or within-user post-selection uncertainty. No classifier is trained and no train/test claim is made.

Every run gets a fresh output folder so previous results cannot be mistaken for the current run. The exported files contain aggregate summaries and provenance only. Invalid supplied data produce an explicit error and a failed-run status.


In [7]:
OUTPUT_BASE = Path("/content/research_project/notebook03_runs") if Path("/content").exists() else Path.cwd()/"notebook03_runs"
RUN_DIR = OUTPUT_BASE / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_STATUS = run_analysis(CONFIG, RUN_DIR)
(RUN_DIR/"public_source_audit.json").write_text(json.dumps(PUBLIC_SOURCE_AUDIT, indent=2), encoding="utf-8")
print(json.dumps(RUN_STATUS, indent=2))
print("Output directory:", RUN_DIR)
if RUN_STATUS["empirical_analysis_executed"]:
    print((RUN_DIR/"sentiment_summary.csv").read_text())
    if (RUN_DIR/"sentiment_contrast.json").exists():
        print((RUN_DIR/"sentiment_contrast.json").read_text())
else:
    print("NO MINDSET FINDINGS PRODUCED. Complete the stated access/input requirement before reporting results.")

{
  "created_at_utc": "2026-11-04T18:20:05.542040+00:00",
  "empirical_analysis_executed": false,
  "status": "NOT_EXECUTED_ACCESS_REQUIRED",
  "next_step": "Obtain the existing MindSET release and Data Use Agreement from its authors; see paper section 7.1."
}
Output directory: /workspace/scratch/0822e5d1c7a5/work/notebook03_runs/20261104T182005541317Z
NO MINDSET FINDINGS PRODUCED. Complete the stated access/input requirement before reporting results.


### Interpretation and report integration

VADER compound sentiment is not an anxiety score. Mentions of algorithms, recommendations or personalisation are an unvalidated keyword indicator; they can concern unrelated contexts. These records cannot be assumed to identify Generation Z, South African residence, perceived personalisation or social comparison. The analysis cannot test H1–H3 or rank platform-caused anxiety.

The existing report correctly records MindSET as unanalysed. Leave that statement in place until an authorised real-data run succeeds. Any later results require a documented methods/results update with the actual release, codebook, selection flow and limitations. Do not copy published authors' metrics into this notebook as if reproduced here.

### Repair and validation record

The earlier version had no interactive upload route or current source check and sampled posts before removing repeated users. This version adds documented access guidance, four streaming input formats, explicit schema and label validation, corrected user-first sampling, aggregate export, file hashing and distinct blocked/failed/executed states.

Software checks used artificial fixtures kept outside this notebook and outside the research package. They tested format consistency, selection counts, duplicates, order invariance, prolific-user handling, invalid labels, missing inputs, VADER summaries, repeatable bootstrap, export privacy and stale-output prevention. Passing these checks does **not** establish validity on the unreleased MindSET corpus. Colab's upload picker was not exercised in a signed-in Colab session. Captured default outputs below/above are access-status checks, not NLP findings.

### Sources

Mankarious, S., Zirikly, A., Wiechmann, D., Kerz, E., Kempa, E. and Qiao, Y. (2025). *MindSET: Advancing mental health benchmarking through large-scale social media data*. arXiv preprint 2511.20672. [Paper, especially section 7.1](https://arxiv.org/html/2511.20672v1#S7.SS1). Access checked 19 October 2026.

[Authors' Hugging Face repository](https://huggingface.co/datasets/smankarious/mindset) and [linked code repository](https://github.com/Zirikly-lab/mindset). Access checked 19 October 2026. The public listing check does not substitute for the authors' access agreement.

Hutto, C.J. and Gilbert, E. (2014). *VADER: A parsimonious rule-based model for sentiment analysis of social media text*. Proceedings of ICWSM, 8(1), pp. 216–225. [https://doi.org/10.1609/icwsm.v8i1.14550](https://doi.org/10.1609/icwsm.v8i1.14550).
